# Buying a theme with ETFs: what you really own, and how it performed

Companion notebook to the Medium post *Buying a Theme Instead of Picking Stocks: The AI Build-Out in Seven ETFs*.
Swap in your own funds in the **Configuration** cell and rerun everything to repeat the analysis for any theme.

> **Not investment advice.** This is an educational, speculative exploration of how a theme *could* be invested in.
> Data comes from Yahoo Finance through the unofficial `yfinance` library and may be incomplete, delayed or wrong.
> Past performance says nothing reliable about the future.

**What it produces**

| Chart | Question |
|---|---|
| Arc diagram | Which layers make up the theme, and which ETF covers each? |
| Sankey | Where does each dollar end up: funds, categories, companies? |
| Overlap heatmap | How much do the funds hold in common? |
| Growth grid | Did each fund's top holdings beat the fund, over a long and a short period? |
| Portfolio growth | What would the whole portfolio be worth today vs the S&P 500? |

**How to run:** Runtime → Run all (Colab) or run cells top to bottom (Jupyter). Data is cached in `data/`, charts are saved to `charts/` as HTML and PNG.

## 0. Setup
Installs anything missing. `yfinance` is always upgraded, because Yahoo changes often and old versions break.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"pandas": "pandas>=2.0", "numpy": "numpy>=1.26", "plotly": "plotly>=5.24",
            "nbformat": "nbformat>=4.2", "kaleido": "kaleido>=1.0"}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance>=0.2.54", *missing])

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab" if IN_COLAB else "Running locally", "| packages ready")

## 1. Configuration
The only cell you need to edit to analyze a different theme.

In [ ]:
# ---------- The theme ----------
THEME = "AI build-out"
ARC_TITLE = "The AI build-out in seven layers, and one ETF for each"
ARC_SUBTITLE = "Arcs show how each layer depends on the next (illustrative)"
ARC_AXIS = ("physical world", "digital world")   # labels under the left and right ends; None to hide

# One ETF per layer, in display order (here from the physical world to the digital one).
# Colors come from STYLE["palette"] in this order.
ETFS = [
    # ticker, layer label
    ("PAVE", "Construction & materials"),
    ("XLU",  "Power generation"),
    ("GRID", "Grid & electrical"),
    ("DTCR", "Data centers"),
    ("SOXQ", "Chips"),
    ("QQQ",  "Cloud & platforms"),
    ("IGV",  "Software & AI apps"),
]
# How the layers depend on each other: (from, to, verb). Your thesis, not data.
ARC_LINKS = [
    ("PAVE", "DTCR", "builds"),
    ("XLU",  "GRID", "generates"),
    ("GRID", "DTCR", "delivers"),
    ("SOXQ", "DTCR", "equips"),
    ("DTCR", "QQQ",  "hosts the cloud"),
    ("QQQ",  "IGV",  "runs the apps"),
]

# ---------- Portfolio ----------
WEIGHTS = None            # None = equal weight, or e.g. {"SOXQ": 50, "QQQ": 10, ...} (normalized to 100%)
INVESTED = 100_000        # starting amount for the portfolio growth chart
BENCHMARK = "SPY"         # S&P 500; None to skip

# ---------- Charts ----------
TOP_N_SANKEY = 10         # top holdings per fund in the Sankey (Yahoo provides up to 10)
TOP_N_PERF = 10           # top holdings per fund in the growth grid
HIGHLIGHT_TOP = 3         # best performers drawn in color in each growth panel; 0 = all grey
LONG_PERIOD = "3Y"        # growth grid, left column:  "YTD", "1Y", "3Y", "5Y"
SHORT_PERIOD = "3M"       # growth grid, right column: "1M", "3M", "6M"
GROWTH_PERIOD = "3Y"      # portfolio growth chart period
START_VALUE = 10_000      # each line in the growth grid starts here

# ---------- Categories (middle column of the Sankey) ----------
CATEGORY_SOURCE = "manual"     # "yahoo"  = Yahoo's industry/sector for each company (works for any theme)
                               # "manual" = your tags below and/or in CATEGORY_FILE
                               # "claude" = Claude proposes tags for untagged companies; you review the file
YAHOO_LEVEL = "industry"       # for "yahoo": "industry" (detailed) or "sector" (broad)
CATEGORY_FILE = "data/categories.csv"   # columns: symbol, category (overrides MANUAL_TAGS); edit freely

# Your theme's categories, in plain English, with a one-line meaning (also used to brief Claude)
CATEGORIES = {
    "Construction, materials & machinery": "Steel, cement, heavy equipment, rail and contractors",
    "Utilities: power plants & lines": "Companies that generate electricity and own the wires",
    "Grid & electrical equipment": "Makers and installers of transformers, cables, switchgear, cooling",
    "Data center landlords & towers": "Owners of data center buildings, towers and fiber",
    "Servers & networking gear": "Servers, switches and routers inside the data centers",
    "Chipmakers": "Chip designers, foundries, memory and chip-making equipment",
    "Big Tech & cloud": "The giant platforms that rent out computing power",
    "Software & AI apps": "Business software and applications that put AI to work",
    "Cybersecurity": "Software that protects networks, data and AI systems",
    "Other: consumer, health, autos": "Holdings not tied to the theme",
}
# Manual tags: category -> tickers (Yahoo symbols; exchange suffixes like .SW are optional)
MANUAL_TAGS = {
    "Construction, materials & machinery": "EMR FAST NUE DE PH UNP NSC ROK CSX URI CRH HWM STLD VMC MLM EME ATI RS AMRZ WWD CAT",
    "Utilities: power plants & lines": "NEE SO DUK CEG AEP D SRE ETR XEL EXC ED VST WEC PEG PCG AEE ATO AWK DTE ES NRG NG EOAN TRN H",
    "Grid & electrical equipment": "ETN PWR JCI HUBB NVT GEV TT ABBN SU PRY",
    "Data center landlords & towers": "EQIX DLR AMT CCI SBAC IRM APLD GDS VNET UNIT NXT AJBU 0788 1686",
    "Servers & networking gear": "CSCO ANET SMCI HPE DELL VRT",
    "Chipmakers": "NVDA AVGO AMD MU INTC MRVL QCOM ADI TSM 2330 KLAC TXN AMAT ASML LRCX MPWR MCHP ON NXPI 000660",
    "Big Tech & cloud": "MSFT AMZN GOOGL GOOG META ORCL AAPL CRWV NBIS",
    "Software & AI apps": "PLTR NOW CRM ADBE DDOG CDNS SNPS APP INTU WDAY ADSK",
    "Cybersecurity": "PANW CRWD FTNT ZS NET",
    "Other: consumer, health, autos": "TSLA COST WMT NFLX AMGN GILD LIN PEP TMUS MSTR",
}
CLAUDE_MODEL = "claude-sonnet-5-5"   # for CATEGORY_SOURCE = "claude"; needs ANTHROPIC_API_KEY

# ---------- Data ----------
HOLDINGS_FILE = None      # optional CSV (fund, symbol, name, weight) replacing Yahoo's top 10 for those funds
EXPENSE_RATIO_OVERRIDE = {}   # e.g. {"QQQ": 0.18} in %, if Yahoo's figure is missing or stale
NAME_OVERRIDES = {}       # e.g. {"BRK-B": "Berkshire"}: short display names
DATA_DIR, OUT_DIR = "data", "charts"
REFRESH = False           # True = download everything again instead of using the cache
SAVE_PNG = True           # also save charts as PNG (needs kaleido + Chrome; falls back to HTML only)

## 2. Style
One place for colors, fonts and sizes. Fund colors are assigned from `palette` in the order of `ETFS`.

In [ ]:
STYLE = {
    "palette": ["#C7CEEA", "#FDFD96", "#FFDAC1", "#B5EAD7", "#AEC6E8", "#FFB7B2", "#E2B6CF",
                "#D4E6B5", "#F6D6AD", "#C9E4DE", "#E8D3F0", "#FFE5B4"],   # pastel, one per fund
    "font": "Inter, Helvetica, Arial, sans-serif",
    "font_size": 12, "title_size": 20,
    "text": "#222222", "muted": "#777777", "grey": "#A0A0A0", "neutral": "#D9D9D9",
    "benchmark": "#444444",
    "template": "plotly_white",
    "dark_factor": 0.55,        # how much darker lines/labels are than their pastel fill
    "link_alpha": 0.60,         # Sankey flow opacity
    "tail_alpha": 0.35,         # Sankey long-tail opacity
    "png_scale": 2,             # PNG resolution multiplier
}

## 3. Helpers

In [ ]:
import json, math, os, re, time, datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import yfinance as yf

FUNDS = [t for t, _ in ETFS]
LAYER = dict(ETFS)
COLOR = {f: STYLE["palette"][i % len(STYLE["palette"])] for i, f in enumerate(FUNDS)}
W = {f: 1 / len(FUNDS) for f in FUNDS} if not WEIGHTS else \
    {f: WEIGHTS.get(f, 0) / sum(WEIGHTS.values()) for f in FUNDS}
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

PERIOD_NAME = {"1M": "1 month", "3M": "3 months", "6M": "6 months", "YTD": "year to date",
               "1Y": "1 year", "3Y": "3 years", "5Y": "5 years"}

# --- colors ---
def darker(hex_color, f=None):
    f = STYLE["dark_factor"] if f is None else f
    h = hex_color.lstrip("#")
    return "rgb({},{},{})".format(*(int(int(h[i:i + 2], 16) * f) for i in (0, 2, 4)))

def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    return "rgba({},{},{},{})".format(*(int(h[i:i + 2], 16) for i in (0, 2, 4)), alpha)

# --- figures ---
def styled(fig, title, subtitle=None, **layout):
    text = title + (f"<br><sup>{subtitle}</sup>" if subtitle else "")
    fig.update_layout(template=STYLE["template"],
                      font=dict(family=STYLE["font"], size=STYLE["font_size"], color=STYLE["text"]),
                      title=dict(text=text, x=0.01, font=dict(size=STYLE["title_size"])), **layout)
    return fig

_png_ok = SAVE_PNG
def save(fig, name):
    global _png_ok
    fig.write_html(os.path.join(OUT_DIR, f"{name}.html"), include_plotlyjs="cdn")
    if _png_ok:
        try:
            fig.write_image(os.path.join(OUT_DIR, f"{name}.png"), scale=STYLE["png_scale"])
        except Exception:
            try:                                   # kaleido >= 1.0 needs Chrome; fetch it once
                import kaleido; kaleido.get_chrome_sync()
                fig.write_image(os.path.join(OUT_DIR, f"{name}.png"), scale=STYLE["png_scale"])
            except Exception as e:
                _png_ok = False
                print(f"PNG export unavailable ({type(e).__name__}); saving HTML only.")
    fig.show()

# --- companies ---
COMPANY_ALIASES = {"GOOG": "GOOGL", "BRK-A": "BRK-B", "2330.TW": "TSM"}   # same company, one id
SHORT_NAMES = {"GOOGL": "Alphabet", "TSM": "TSMC", "META": "Meta", "AVGO": "Broadcom", "NVDA": "Nvidia",
               "MSFT": "Microsoft", "AMZN": "Amazon", "AAPL": "Apple", "AMD": "AMD", "PLTR": "Palantir",
               **NAME_OVERRIDES}
_SUFFIX = (r"(Inc|Corp|Corporation|Company|Co|plc|Ltd|Limited|SA|SE|AG|NV|N\.V|S\.p\.A|SpA|Holdings?|"
           r"Group|Class [A-C]|ADR|Ordinary Shares|Common Stock|Trust)")

def company_id(symbol):
    s = str(symbol).upper().strip()
    return COMPANY_ALIASES.get(s, s)

def base_symbol(symbol):
    s = str(symbol).upper().strip()
    return s.split(":")[-1].split(".")[0]

def short_name(cid, name):
    if cid in SHORT_NAMES:
        return SHORT_NAMES[cid]
    n = re.sub(r"\s+-\s*.*$|\s+-$", "", str(name))          # drop " - Class A ..." tails
    for _ in range(4):
        n = re.sub(rf",?\s+{_SUFFIX}\.?$", "", n.strip(" ,."), flags=re.IGNORECASE)
    return n.strip(" ,.&") or cid

def wrap(text, width=28):
    words, lines, line = str(text).split(), [], ""
    for w_ in words:
        if line and len(line) + len(w_) > width:
            lines.append(line); line = w_
        else:
            line = f"{line} {w_}".strip()
    return "<br>".join(lines + [line])

# --- periods and growth ---
def period_start(index, end, p):
    if p == "YTD":
        return index[index.year < end.year].max()            # last close of last year
    n, unit = int(p[:-1]), p[-1]
    return end - (pd.DateOffset(months=n) if unit == "M" else pd.DateOffset(years=n))

def growth(series, start, end, base):
    s = series.dropna()
    s = s[(s.index >= start) & (s.index <= end)]
    if s.empty or (s.index[0] - start).days > 7:              # not enough history
        return None
    return base * s / s.iloc[0]

def retry(fn, tries=3, wait=2):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            if i == tries - 1:
                print(f"   failed: {e}")
                return None
            time.sleep(wait * (i + 1))

def spread(values, gap):
    # Nudge label positions apart so names don't overlap; keeps the stack centered on the data
    order = np.argsort(values)
    pos = np.array(values, dtype=float)[order]
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    out = np.empty_like(pos)
    out[order] = pos - max(0, (pos.max() - np.max(values)) / 2)
    return out

## 4. Data from Yahoo Finance
Downloads fund facts, top holdings, company details and dividend-adjusted prices, then caches them in `data/`.
Set `REFRESH = True` to download again. Non-US stocks are converted to US dollars.

In [ ]:
def cached(name, builder, **read_kw):
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path) and not REFRESH:
        return pd.read_csv(path, **read_kw)
    df = builder()
    df.to_csv(path, index=read_kw.get("index_col") is not None)
    return pd.read_csv(path, **read_kw)

def fetch_funds():
    rows = []
    for f in FUNDS:
        t = yf.Ticker(f)
        info = retry(lambda: t.info) or {}
        rows.append({"fund": f, "name": info.get("longName") or info.get("shortName") or f,
                     "issuer": info.get("fundFamily", ""),
                     "expense_ratio_pct": info.get("netExpenseRatio", np.nan),
                     "total_assets": info.get("totalAssets", np.nan),
                     "yield_pct": (info.get("yield") or np.nan) * 100})
        time.sleep(0.5)
    return pd.DataFrame(rows)

def fetch_holdings():
    rows = []
    for f in FUNDS:
        top = retry(lambda: yf.Ticker(f).funds_data.top_holdings)
        if top is None or top.empty:
            print(f"   {f}: no holdings from Yahoo")
            continue
        top = top.reset_index()
        top.columns = [str(c).strip().lower().replace(" ", "_") for c in top.columns]
        for sym, name, w_ in zip(top["symbol"], top["name"], top["holding_percent"].astype(float)):
            rows.append({"fund": f, "symbol": sym, "name": name, "weight": w_})
        time.sleep(0.5)
    return pd.DataFrame(rows)

FUND_INFO = cached("funds.csv", fetch_funds).set_index("fund")
FUND_INFO["expense_ratio_pct"] = [EXPENSE_RATIO_OVERRIDE.get(f, er) for f, er in FUND_INFO["expense_ratio_pct"].items()]
HOLDINGS = cached("holdings.csv", fetch_holdings)
if HOLDINGS_FILE:                                     # optional full holdings for some funds
    custom = pd.read_csv(HOLDINGS_FILE)
    custom.columns = [c.strip().lower() for c in custom.columns]
    HOLDINGS = pd.concat([HOLDINGS[~HOLDINGS["fund"].isin(custom["fund"])], custom[["fund", "symbol", "name", "weight"]]])
HOLDINGS["weight"] = HOLDINGS["weight"].astype(float)
HOLDINGS.loc[HOLDINGS.groupby("fund")["weight"].transform("max") > 1.5, "weight"] /= 100   # % -> fraction
HOLDINGS["symbol"] = HOLDINGS["symbol"].astype(str).str.upper().str.strip()
HOLDINGS["company"] = HOLDINGS["symbol"].map(company_id)

# Company details (sector, industry, currency): fetched once per symbol, cached incrementally
comp_path = os.path.join(DATA_DIR, "companies.csv")
COMPANIES = pd.read_csv(comp_path) if os.path.exists(comp_path) and not REFRESH else \
            pd.DataFrame(columns=["symbol", "sector", "industry", "currency"])
todo = sorted(set(HOLDINGS["symbol"]) - set(COMPANIES["symbol"]))
if todo:
    print(f"Looking up {len(todo)} companies ...")
    new = []
    for s in todo:
        info = retry(lambda: yf.Ticker(s).info) or {}
        new.append({"symbol": s, "sector": info.get("sector", ""), "industry": info.get("industry", ""),
                    "currency": info.get("currency", "USD")})
        time.sleep(0.3)
    COMPANIES = pd.concat([COMPANIES, pd.DataFrame(new)], ignore_index=True)
    COMPANIES.to_csv(comp_path, index=False)
COMPANIES = COMPANIES.set_index("symbol")

# Prices: ~6 years, dividend-adjusted
START = (pd.Timestamp.today() - pd.DateOffset(years=6)).date()
def download(tickers):
    def go_():
        df = yf.download(sorted(set(tickers)), start=START, auto_adjust=True, progress=False)["Close"]
        return df.to_frame(tickers[0]) if isinstance(df, pd.Series) else df
    return lambda: retry(go_)

FUND_PX = cached("prices_funds.csv", download(FUNDS + ([BENCHMARK] if BENCHMARK else [])),
                 index_col=0, parse_dates=True)
STOCK_PX = cached("prices_stocks.csv", download(list(HOLDINGS["symbol"].unique())), index_col=0, parse_dates=True)

# Convert non-US listings to USD (GBp pence counts as GBP: only the ratio matters for returns)
ccy = COMPANIES["currency"].fillna("USD").astype(str).str.upper().replace({"GBX": "GBP"})
foreign = {s: c for s, c in ccy.items() if c not in ("USD", "") and s in STOCK_PX}
if foreign:
    FX = cached("fx.csv", download([f"{c}USD=X" for c in set(foreign.values())]), index_col=0, parse_dates=True)
    for s, c in foreign.items():
        STOCK_PX[s] = STOCK_PX[s] * FX[f"{c}USD=X"].reindex(STOCK_PX.index).ffill()

END = FUND_PX[FUNDS].dropna(how="all").index.max()

# ---------- Data check ----------
check = pd.DataFrame(index=FUNDS)
check["holdings"] = HOLDINGS.groupby("fund").size()
check["top_holdings_weight"] = HOLDINGS.groupby("fund")["weight"].sum()
check["expense_ratio_pct"] = FUND_INFO["expense_ratio_pct"]
check["first_price"] = [FUND_PX[f].first_valid_index().date() for f in FUNDS]
check["companies_missing_prices"] = [
    ", ".join(s for s in HOLDINGS.loc[HOLDINGS["fund"] == f, "symbol"]
              if s not in STOCK_PX or STOCK_PX[s].dropna().empty) for f in FUNDS]
print(f"Data as of {END:%b %d, %Y}")
display(check.style.format({"top_holdings_weight": "{:.0%}", "expense_ratio_pct": "{:.2f}%"}, na_rep="n/a"))

## 5. Fund summary

In [ ]:
def total_return(series, p):
    g = growth(series, period_start(FUND_PX.index, END, p), END, 1.0)
    return g.iloc[-1] - 1 if g is not None else np.nan

SUMMARY = FUND_INFO[["name", "issuer", "expense_ratio_pct", "total_assets", "yield_pct"]].copy()
SUMMARY.insert(0, "layer", [LAYER[f] for f in SUMMARY.index])
for p in ["1M", "3M", "YTD", "1Y", "3Y", "5Y"]:
    SUMMARY[p] = [total_return(FUND_PX[f], p) for f in SUMMARY.index]
SUMMARY.to_csv(os.path.join(OUT_DIR, "fund_summary.csv"))
display(SUMMARY.style.format({"expense_ratio_pct": "{:.2f}%", "total_assets": lambda v: f"${v / 1e9:,.1f}B",
                              "yield_pct": "{:.2f}%", **{p: "{:+.1%}" for p in ["1M", "3M", "YTD", "1Y", "3Y", "5Y"]}},
                             na_rep="n/a"))

## 6. Categories for the Sankey
Every company in the top holdings gets one category. Three ways to do it:

- **`yahoo`**: Yahoo's industry (or sector). Automatic, works for any theme, but generic.
- **`manual`**: your own tags, from `MANUAL_TAGS` and/or `data/categories.csv` (the file wins). Untagged companies show as *Unclassified*.
- **`claude`**: Claude proposes a category for each untagged company, choosing only from your `CATEGORIES`, with a confidence and a reason. Proposals are written to `data/categories.csv`, where you review and edit them; the next run uses your edited file and only asks Claude about new companies.

Every run also writes `data/categories_review.csv` with each company's category, where it came from, and Yahoo's industry, to make reviewing easy.

In [ ]:
COMP = (HOLDINGS.groupby("company")
                .agg(symbol=("symbol", "first"), name=("name", "first"), funds=("fund", lambda s: ", ".join(sorted(set(s))))))
COMP["label"] = [short_name(c, n) for c, n in zip(COMP.index, COMP["name"])]
COMP["industry"] = COMP["symbol"].map(COMPANIES["industry"]).fillna("")
COMP["sector"] = COMP["symbol"].map(COMPANIES["sector"]).fillna("")

def lookup(tags, symbol):
    return tags.get(str(symbol).upper()) or tags.get(base_symbol(symbol))

def read_tag_file():
    if not os.path.exists(CATEGORY_FILE):
        return {}
    t = pd.read_csv(CATEGORY_FILE).dropna(subset=["category"])
    return {str(s).upper(): c for s, c in zip(t["symbol"], t["category"]) if str(c).strip()}

def ask_claude(companies):
    # Claude proposes one category per company, from CATEGORIES only, via a forced tool call.
    import importlib.util as iu
    if iu.find_spec("anthropic") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic>=0.40"])
    import anthropic
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key and IN_COLAB:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    client = anthropic.Anthropic(api_key=key)
    tool = {"name": "record_categories", "description": "Record one category per company.",
            "input_schema": {"type": "object", "required": ["items"], "properties": {"items": {
                "type": "array", "items": {"type": "object", "required": ["symbol", "category", "confidence", "reason"],
                "properties": {"symbol": {"type": "string"},
                               "category": {"type": "string", "enum": list(CATEGORIES)},
                               "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                               "reason": {"type": "string", "description": "One short sentence."}}}}}}}
    cats = "\n".join(f"- {c}: {d}" for c, d in CATEGORIES.items())
    lst = "\n".join(f"- {r.symbol}: {r.name} (Yahoo industry: {r.industry or 'n/a'}; held by {r.funds})"
                    for r in companies.itertuples())
    prompt = (f"Theme: {THEME}.\nAssign each company to exactly one category, based on where most of its "
              f"revenue comes from.\n\nCategories:\n{cats}\n\nCompanies:\n{lst}\n\n"
              "If unsure, still choose the best category but give a confidence below 0.7.")
    resp = client.messages.create(model=CLAUDE_MODEL, max_tokens=4000, tools=[tool],
                                  tool_choice={"type": "tool", "name": "record_categories"},
                                  messages=[{"role": "user", "content": prompt}])
    items = next(b for b in resp.content if b.type == "tool_use").input["items"]
    return pd.DataFrame(items)

if CATEGORY_SOURCE == "yahoo":
    COMP["category"] = COMP["industry" if YAHOO_LEVEL == "industry" else "sector"].replace("", "Unclassified")
    COMP["source"] = "yahoo"
else:
    tags = {s.upper(): c for c, syms in MANUAL_TAGS.items() for s in syms.split()}
    file_tags = read_tag_file()
    COMP["category"] = [lookup(file_tags, s) or lookup(tags, s) for s in COMP["symbol"]]
    COMP["source"] = ["file" if lookup(file_tags, s) else "manual" if c else None
                      for s, c in zip(COMP["symbol"], COMP["category"])]
    untagged = COMP[COMP["category"].isna()]
    if CATEGORY_SOURCE == "claude" and len(untagged):
        print(f"Asking Claude ({CLAUDE_MODEL}) to tag {len(untagged)} companies ...")
        prop = ask_claude(untagged)
        prop["symbol"] = prop["symbol"].str.upper()
        old = pd.read_csv(CATEGORY_FILE) if os.path.exists(CATEGORY_FILE) else pd.DataFrame()
        new = prop.assign(source="claude", reviewed=False)[["symbol", "category", "confidence", "reason", "source", "reviewed"]]
        pd.concat([old, new]).drop_duplicates("symbol", keep="last").to_csv(CATEGORY_FILE, index=False)
        print(f"Saved proposals to {CATEGORY_FILE}. Review them (low confidence first), edit if needed, rerun.")
        display(new.sort_values("confidence").style.format({"confidence": "{:.2f}"}).hide(axis="index"))
        file_tags = read_tag_file()
        COMP["category"] = [lookup(file_tags, s) or c for s, c in zip(COMP["symbol"], COMP["category"])]
        COMP["source"] = ["claude" if s.upper() in set(new["symbol"]) else src for s, src in zip(COMP["symbol"], COMP["source"])]
    COMP["category"] = COMP["category"].fillna("Unclassified")
    COMP["source"] = COMP["source"].fillna("untagged")

COMP[["symbol", "label", "category", "source", "industry", "funds"]].to_csv(
    os.path.join(DATA_DIR, "categories_review.csv"))
print(COMP["source"].value_counts().to_string())
n_un = (COMP["category"] == "Unclassified").sum()
if n_un:
    print(f"\n{n_un} companies are Unclassified: {', '.join(COMP.loc[COMP['category'] == 'Unclassified', 'label'])}")
    print(f"Tag them in {CATEGORY_FILE} (columns: symbol, category) or use CATEGORY_SOURCE = 'claude'.")

## 7. Chart 0: the theme in layers (arc diagram)

In [ ]:
def arc_chart():
    pos = {f: i for i, f in enumerate(FUNDS)}
    rx, ry = 0.21, 0.178                                  # circle radius in data units
    ends = {}
    for i, (a, b, _) in enumerate(ARC_LINKS):
        ends.setdefault(a, []).append((pos[b], i, "s"))
        ends.setdefault(b, []).append((pos[a], i, "t"))
    anchor = {}                                           # spread arc endpoints over each circle
    for node, items in ends.items():
        items.sort()
        for k, (_, i, role) in enumerate(items):
            dx = 0 if len(items) == 1 else -0.13 + 0.26 * k / (len(items) - 1)
            anchor[(i, role)] = (pos[node] + dx, ry * math.sqrt(max(0, 1 - (dx / rx) ** 2)))

    fig = go.Figure()
    fig.add_shape(type="line", x0=-0.4, x1=len(FUNDS) - 0.6, y0=0, y1=0, layer="below",
                  line=dict(color=STYLE["neutral"], width=1))
    for i, (a, b, verb) in enumerate(ARC_LINKS):
        (x1, y1), (x2, y2) = anchor[(i, "s")], anchor[(i, "t")]
        h = 0.30 + 0.36 * abs(pos[b] - pos[a])
        t = np.linspace(0, np.pi, 80)
        xs = (x1 + x2) / 2 - (x2 - x1) / 2 * np.cos(t)
        ys = y1 + (y2 - y1) * (1 - np.cos(t)) / 2 + h * np.sin(t)
        c = darker(COLOR[a])
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=c, width=3), opacity=0.85,
                                 hoverinfo="skip", showlegend=False))
        fig.add_annotation(x=x2, y=y2, ax=x2, ay=y2 + 0.13, xref="x", yref="y", axref="x", ayref="y", text="",
                           showarrow=True, arrowhead=2, arrowsize=1.3, arrowwidth=2.5, arrowcolor=c)
        fig.add_annotation(x=(x1 + x2) / 2, y=(y1 + y2) / 2 + h, text=f"<i>{verb}</i>", showarrow=False,
                           yshift=11, font=dict(size=13, color=c), bgcolor="rgba(255,255,255,0.85)")
    fig.add_trace(go.Scatter(
        x=list(range(len(FUNDS))), y=[0] * len(FUNDS), mode="markers+text", showlegend=False,
        marker=dict(size=74, color=[COLOR[f] for f in FUNDS], line=dict(color=[darker(COLOR[f]) for f in FUNDS], width=2)),
        text=[f"<b>{f}</b>" for f in FUNDS], textfont=dict(size=16, color=STYLE["text"]),
        hovertext=[f"<b>{f}</b> · {LAYER[f]}<br>{FUND_INFO.loc[f, 'name']}" for f in FUNDS], hoverinfo="text"))
    for f in FUNDS:
        x, er = pos[f], FUND_INFO.loc[f, "expense_ratio_pct"]
        fig.add_annotation(x=x, y=-0.22, text=f"<b>{LAYER[f]}</b>", showarrow=False,
                           font=dict(size=14, color=darker(COLOR[f], 0.45)))
        fig.add_annotation(x=x, y=-0.38, text=wrap(FUND_INFO.loc[f, "name"]), showarrow=False,
                           font=dict(size=11, color="#555"))
        fig.add_annotation(x=x, y=-0.56, text=f"Expense ratio {er:.2f}%", showarrow=False,
                           font=dict(size=11, color="#888"))
    n = len(FUNDS) - 1
    if ARC_AXIS:
        fig.add_annotation(x=n, y=-0.72, ax=0, ay=-0.72, xref="x", yref="y", axref="x", ayref="y", text="",
                           showarrow=True, arrowhead=2, arrowwidth=1.2, arrowcolor="#bbb")
        for x, txt, anchor in [(0, ARC_AXIS[0], "left"), (n, ARC_AXIS[1], "right")]:
            fig.add_annotation(x=x, y=-0.80, text=txt, showarrow=False, xanchor=anchor,
                               font=dict(size=11, color="#999"))
    styled(fig, ARC_TITLE, ARC_SUBTITLE, width=180 * len(FUNDS), height=640, plot_bgcolor="white",
           margin=dict(l=20, r=20, t=90, b=20),
           xaxis=dict(visible=False, range=[-0.6, n + 0.6]), yaxis=dict(visible=False, range=[-0.85, 1.65]))
    save(fig, "chart0_arc")

arc_chart()

## 8. Chart 1: what you actually own (Sankey) and fund overlap

In [ ]:
# One row per fund x company, weighted by the portfolio weights
H = (HOLDINGS.groupby(["fund", "company"], as_index=False)["weight"].sum()
             .assign(rank=lambda d: d.groupby("fund")["weight"].rank(ascending=False, method="first")))
TOP = H[H["rank"] <= TOP_N_SANKEY].copy()
TOP["label"] = TOP["company"].map(COMP["label"])
TOP["category"] = TOP["company"].map(COMP["category"])
TOP["exposure"] = TOP["fund"].map(W) * TOP["weight"]
tail = pd.Series({f: W[f] * max(0, 1 - TOP.loc[TOP["fund"] == f, "weight"].sum()) for f in FUNDS})

by_co = TOP.groupby(["company", "label", "category"], as_index=False).agg(
    funds=("fund", lambda s: ", ".join(s)), n_funds=("fund", "nunique"), exposure=("exposure", "sum"))
shared = by_co[by_co["n_funds"] > 1].sort_values("exposure", ascending=False)
print(f"{len(FUNDS)} funds x top {TOP_N_SANKEY} = {len(TOP)} slots -> {len(by_co)} distinct companies")
print(f"Top holdings cover {TOP['exposure'].sum():.1%} of the portfolio; the long tail holds {tail.sum():.1%}.")
print(f"{len(shared)} companies are held by more than one fund ({shared['exposure'].sum():.1%} of the portfolio):")
display(shared[["label", "category", "funds", "exposure"]].style.format({"exposure": "{:.2%}"}).hide(axis="index"))

def sankey():
    # Order categories by the average position of the funds feeding them, which keeps flows from crossing
    fpos = {f: i for i, f in enumerate(FUNDS)}
    cat_rank = TOP.assign(p=TOP["fund"].map(fpos) * TOP["exposure"]).groupby("category").apply(
        lambda d: d["p"].sum() / d["exposure"].sum(), include_groups=False).sort_values()
    TAIL_CAT, TAIL_CO = f"Beyond each fund's top {TOP_N_SANKEY}", "All other holdings"
    cats = list(cat_rank.index) + [TAIL_CAT]
    cos = (by_co.assign(r=by_co["category"].map({c: i for i, c in enumerate(cats)}))
                .sort_values(["r", "exposure"], ascending=[True, False])["label"].tolist()) + [TAIL_CO]
    labels = FUNDS + cats + cos
    idx = {n: i for i, n in enumerate(labels)}

    src, tgt, val, col = [], [], [], []
    def link(a, b, v, c):
        src.append(idx[a]); tgt.append(idx[b]); val.append(v); col.append(c)
    for r in TOP.groupby(["fund", "category"], as_index=False)["exposure"].sum().itertuples():
        link(r.fund, r.category, r.exposure, rgba(COLOR[r.fund], STYLE["link_alpha"]))
    for r in TOP.itertuples():                 # one band per fund, so shared companies show mixed colors
        link(r.category, r.label, r.exposure, rgba(COLOR[r.fund], STYLE["link_alpha"]))
    for f, v in tail.items():
        link(f, TAIL_CAT, v, rgba(COLOR[f], STYLE["tail_alpha"]))
        link(TAIL_CAT, TAIL_CO, v, rgba(COLOR[f], STYLE["tail_alpha"]))

    # Pin node positions in our order (Plotly would otherwise reorder them)
    value = ([W[f] for f in FUNDS] +
             [TOP.loc[TOP["category"] == c, "exposure"].sum() for c in cats[:-1]] + [tail.sum()] +
             [by_co.set_index("label")["exposure"].get(c, 0) for c in cos[:-1]] + [tail.sum()])
    def stack(vals, gap):
        total, ys, cum = sum(vals) + gap * (len(vals) - 1), [], 0.0
        for v in vals:
            ys.append(min(0.999, max(0.001, (cum + v / 2) / total))); cum += v + gap
        return ys
    n1, n2, n3 = len(FUNDS), len(cats), len(cos)
    height, pad = max(900, 22 * n3), 4
    # Plotly sizes nodes from the busiest column; matching its spacing keeps nodes from being pushed around
    plot_h = height - 110
    gap = 0.9 * pad / (plot_h - (n3 - 1) * pad)
    x = [0.001] * n1 + [0.42] * n2 + [0.999] * n3
    y = stack(value[:n1], 0.03) + stack(value[n1:n1 + n2], 0.02) + stack(value[n1 + n2:], gap)

    fig = go.Figure(go.Sankey(
        arrangement="snap", valueformat=".1%",
        node=dict(label=labels, x=x, y=y, pad=pad, thickness=14, line=dict(color="white", width=0.5),
                  color=[COLOR[f] for f in FUNDS] + [STYLE["neutral"]] * (n2 + len(cos)),
                  hovertemplate="%{label}<br>%{value:.1%} of portfolio<extra></extra>"),
        link=dict(source=src, target=tgt, value=val, color=col,
                  hovertemplate="%{source.label} → %{target.label}<br>%{value:.2%}<extra></extra>")))
    weights = "equal weights" if not WEIGHTS else "custom weights"
    styled(fig, f"Where $1 in {len(FUNDS)} {THEME} ETFs ends up",
           f"{weights} · top {TOP_N_SANKEY} holdings per fund · data as of {END:%b %d, %Y}",
           height=height, width=1300, margin=dict(l=20, r=20, t=90, b=20))
    save(fig, "chart1_sankey")

def overlap():
    M = H.pivot_table(index="company", columns="fund", values="weight", aggfunc="sum").reindex(columns=FUNDS).fillna(0)
    O = pd.DataFrame([[np.minimum(M[a], M[b]).sum() if a != b else np.nan for b in FUNDS] for a in FUNDS],
                     index=FUNDS, columns=FUNDS)
    fig = px.imshow(O, text_auto=".0%", zmin=0, color_continuous_scale=["#FFFFFF", darker(STYLE["palette"][4], 0.8)])
    styled(fig, "How much two funds hold in common", "sum of the smaller weight over shared top holdings",
           width=640, height=520, coloraxis_colorbar_tickformat=".0%",
           xaxis=dict(showgrid=False), yaxis=dict(showgrid=False))
    save(fig, "chart1b_overlap")

sankey()
overlap()

## 9. Chart 2: each fund vs its top holdings, long term vs short term
$10,000 in each fund (shaded band) vs $10,000 in each of its current top holdings (lines). The best `HIGHLIGHT_TOP` are in color.
Members are **today's** top holdings, so long periods flatter them (they became top holdings partly by rising).

In [ ]:
def growth_grid():
    periods = [LONG_PERIOD, SHORT_PERIOD]
    starts = {p: period_start(FUND_PX.index, END, p) for p in periods}
    members = (H[H["rank"] <= TOP_N_PERF].assign(label=lambda d: d["company"].map(COMP["label"]),
                                                  symbol=lambda d: d["company"].map(COMP["symbol"])))
    panels, rows = {}, []
    for f in FUNDS:
        for p in periods:
            fg = growth(FUND_PX[f], starts[p], END, START_VALUE)
            lines, skipped = [], []
            for r in members[members["fund"] == f].itertuples():
                g = growth(STOCK_PX[r.symbol], starts[p], END, START_VALUE) if r.symbol in STOCK_PX else None
                (lines.append((r.label, g)) if g is not None else skipped.append(r.label))
            lines.sort(key=lambda t: t[1].iloc[-1], reverse=True)
            beat = sum(g.iloc[-1] > fg.iloc[-1] for _, g in lines)
            panels[(f, p)] = (fg, lines, beat)
            rows.append({"fund": f, "period": PERIOD_NAME[p], "fund_return": fg.iloc[-1] / START_VALUE - 1,
                         "beat_fund": f"{beat} of {len(lines)}", "top_3": ", ".join(l for l, _ in lines[:3]),
                         "bottom_3": ", ".join(l for l, _ in lines[-3:]), "excluded": ", ".join(skipped)})

    pct = lambda v: f"{v / START_VALUE - 1:+.0%}"
    titles = [f"<b>{f}</b> · {PERIOD_NAME[p]}: fund {pct(panels[(f, p)][0].iloc[-1])}, "
              f"{panels[(f, p)][2]} of {len(panels[(f, p)][1])} beat it" for f in FUNDS for p in periods]
    fig = make_subplots(rows=len(FUNDS), cols=2, shared_xaxes="columns", horizontal_spacing=0.16,
                        vertical_spacing=0.03, subplot_titles=titles)
    for a in fig.layout.annotations:
        a.update(font_size=12)
    for row, f in enumerate(FUNDS, 1):
        for col, p in enumerate(periods, 1):
            fg, lines, _ = panels[(f, p)]
            n = (row - 1) * 2 + col
            yref, xname = ("y" if n == 1 else f"y{n}"), ("xaxis" if n == 1 else f"xaxis{n}")
            dark, log = darker(COLOR[f]), p in ("3Y", "5Y")
            fig.add_trace(go.Scatter(x=fg.index, y=[START_VALUE] * len(fg), line=dict(width=0),
                                     hoverinfo="skip", showlegend=False), row=row, col=col)
            fig.add_trace(go.Scatter(x=fg.index, y=fg, fill="tonexty", fillcolor=COLOR[f], showlegend=False,
                                     line=dict(color=dark, width=2.5),
                                     hovertemplate=f"<b>{f}</b> %{{x|%b %d, %Y}}<br>$%{{y:,.0f}}<extra></extra>"),
                          row=row, col=col)
            for k, (lbl, g) in enumerate(lines):
                top = k < HIGHLIGHT_TOP
                fig.add_trace(go.Scatter(x=g.index, y=g, showlegend=False, opacity=1 if top else 0.6,
                                         line=dict(color=dark if top else STYLE["grey"], width=1.6 if top else 0.9),
                                         hovertemplate=f"{lbl} %{{x|%b %d, %Y}}<br>$%{{y:,.0f}}<extra></extra>"),
                              row=row, col=col)
            fig.add_hline(y=START_VALUE, line=dict(color="grey", width=0.8, dash="dot"), row=row, col=col)
            fig.update_yaxes(type="log" if log else "linear", tickprefix="$", tickformat="~s", row=row, col=col)
            names = ([(f"<b>{f} fund</b> {pct(fg.iloc[-1])}", fg.iloc[-1], "black", 10)] +
                     [(f"{l} {pct(g.iloc[-1])}", g.iloc[-1], dark if k < HIGHLIGHT_TOP else STYLE["muted"], 9)
                      for k, (l, g) in enumerate(lines)])
            vals = [math.log10(v) if log else v for _, v, _, _ in names]
            gap = ((max(vals) - min(vals)) or 1) * 0.085
            x_label = fig.layout[xname].domain[1] + 0.004
            for (text, _, c, size), yv in zip(names, spread(vals, gap)):
                fig.add_annotation(x=x_label, xref="paper", y=yv, yref=yref, text=text, showarrow=False,
                                   xanchor="left", font=dict(size=size, color=c))
    styled(fig, f"Growth of ${START_VALUE:,.0f}: each fund vs its current top {TOP_N_PERF} holdings",
           f"left: {PERIOD_NAME[LONG_PERIOD]} from {starts[LONG_PERIOD]:%b %d, %Y} · right: {PERIOD_NAME[SHORT_PERIOD]} "
           f"from {starts[SHORT_PERIOD]:%b %d, %Y} · to {END:%b %d, %Y} · dividends reinvested, in USD · "
           f"best {HIGHLIGHT_TOP} in color", height=330 * len(FUNDS), width=1500,
           margin=dict(l=60, r=170, t=110, b=40))
    save(fig, "chart2_growth_grid")

    res = pd.DataFrame(rows).set_index(["fund", "period"])
    res.to_csv(os.path.join(OUT_DIR, "chart2_summary.csv"))
    display(res.style.format({"fund_return": "{:+.1%}"}))
    print("Still a top-3 performer in both periods:")
    for f in FUNDS:
        both = {l for l, _ in panels[(f, LONG_PERIOD)][1][:3]} & {l for l, _ in panels[(f, SHORT_PERIOD)][1][:3]}
        print(f"  {f:5} {', '.join(sorted(both)) or 'none'}")

growth_grid()

## 10. Chart 3: what the portfolio would be worth today

In [ ]:
def portfolio_growth():
    def sleeves(start):
        win = FUND_PX.loc[FUND_PX.index >= start, FUNDS].loc[:END]
        return None if win.iloc[0].isna().any() else INVESTED * pd.Series(W) * win / win.iloc[0]

    rows = []
    for p in ["1M", "3M", "6M", "YTD", "1Y", "3Y", "5Y"]:
        start = period_start(FUND_PX.index, END, p)
        s = sleeves(start)
        if s is None:
            continue
        v, yrs = s.iloc[-1].sum(), (END - s.index[0]).days / 365.25
        row = {"period": PERIOD_NAME[p], "start": s.index[0].date(), "value": v, "total_return": v / INVESTED - 1,
               "annualized": (v / INVESTED) ** (1 / yrs) - 1 if yrs >= 1 else np.nan,
               "best_fund": s.iloc[-1].idxmax(), "worst_fund": s.iloc[-1].idxmin()}
        if BENCHMARK:
            b = FUND_PX[BENCHMARK].loc[s.index]
            row[f"{BENCHMARK}_value"] = INVESTED * b.iloc[-1] / b.iloc[0]
            row["difference"] = v - row[f"{BENCHMARK}_value"]
        rows.append(row)
    res = pd.DataFrame(rows).set_index("period")
    res.to_csv(os.path.join(OUT_DIR, "chart3_portfolio_returns.csv"))
    money = {c: "${:,.0f}" for c in ["value", f"{BENCHMARK}_value", "difference"] if c in res}
    display(res.style.format({**money, "total_return": "{:+.1%}", "annualized": "{:+.1%}"}, na_rep="n/a"))

    s = sleeves(period_start(FUND_PX.index, END, GROWTH_PERIOD))
    end_vals = s.iloc[-1].sort_values()
    fig = go.Figure()
    for f in end_vals.index:
        fig.add_trace(go.Scatter(x=s.index, y=s[f], stackgroup="one", name=f, fillcolor=COLOR[f],
                                 line=dict(width=0.5, color=COLOR[f]),
                                 hovertemplate=f"{f} %{{x|%b %d, %Y}}<br>$%{{y:,.0f}}<extra></extra>"))
    if BENCHMARK:
        b = FUND_PX[BENCHMARK].loc[s.index]
        fig.add_trace(go.Scatter(x=b.index, y=INVESTED * b / b.iloc[0], name=f"{BENCHMARK} (benchmark)",
                                 line=dict(color=STYLE["benchmark"], width=2, dash="dash")))
    fig.add_hline(y=INVESTED, line=dict(color="grey", width=0.8, dash="dot"))
    cum = 0
    for f in end_vals.index:
        fig.add_annotation(x=1.005, xref="paper", y=cum + end_vals[f] / 2, yref="y", xanchor="left", showarrow=False,
                           text=f"{f} ${end_vals[f] / 1000:,.1f}k", font=dict(size=11))
        cum += end_vals[f]
    fig.add_annotation(x=1.005, xref="paper", y=cum, yref="y", xanchor="left", yshift=12, showarrow=False,
                       text=f"<b>Total ${cum / 1000:,.1f}k</b>", font=dict(size=12))
    weights = "split equally" if not WEIGHTS else "with custom weights"
    styled(fig, f"${INVESTED:,.0f} {weights} across {len(FUNDS)} {THEME} ETFs, {PERIOD_NAME[GROWTH_PERIOD]}",
           f"{s.index[0]:%b %d, %Y} to {END:%b %d, %Y} · bought and held, dividends reinvested · each band is one fund",
           height=560, width=1050, margin=dict(l=60, r=150, t=90, b=40), hovermode="x unified",
           yaxis=dict(tickprefix="$", tickformat="~s"), legend=dict(orientation="h", y=-0.12))
    save(fig, "chart3_portfolio_growth")
    print("Weights at the end (no rebalancing):")
    print((end_vals[::-1] / end_vals.sum()).map("{:.1%}".format).to_string())

portfolio_growth()
print(f"\nAll charts and tables saved to '{OUT_DIR}/'.")